# 14 · `gl_engine/schema/validate.py`

## What this file is for

Check a submission against ISO's declared schema **before** rating it — and report *findings*, not exceptions.

Two decisions drive the whole file. A submission with three problems should report three, not the first one. And validation must never be what decides whether a rating happens: the engine's own refusals are stricter and better placed. This tells a caller what ISO would object to.

**Depends on:** [`13-schema-fields`](13-schema-fields.ipynb), [`09-interp-tree`](09-interp-tree.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.schema import validate

for name, obj in vars(validate).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != validate.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Validate a real submission.

In [ ]:
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook

book = ResolvedBook(EditionResolver().resolve("GA", "20260811"))

import json
from pathlib import Path
from gl_engine.schema.fields import Schema
from gl_engine.schema.validate import validate

payload = json.loads(Path("../Engine_Payloads/GA/submission.json").read_text())
schema  = Schema.for_book(book)

findings = validate(payload, schema)
print(f"{len(findings)} finding(s)")
for f in findings[:8]:
    print("  ", f)

## The interesting case

### Five checks, and each says what it does *not* cover

In [ ]:
import sys

# `gl_engine.schema` re-exports the validate FUNCTION under the module's own
# name, so `import gl_engine.schema.validate as V` would hand you the function.
V = sys.modules["gl_engine.schema.validate"]

print(V.__doc__[:1400])

The honesty in those descriptions is the point. **V1 is a warning rather than an error** because ISO's own request format carries envelope fields the form doesn't declare. **V2 skips conditionally-required fields** because the condition dialect isn't implemented. **V3 says whether its answer was exact or a safe superset.**

A validator that reported confidently on all five would be wrong more often and you'd never know which finding to trust.

### Severity is a first-class distinction

In [ ]:
print("ERROR  :", V.ERROR)
print("WARNING:", V.WARNING)
print("INFO   :", V.INFO)
print()
from collections import Counter
print("this submission:", dict(Counter(getattr(f, "severity", "?") for f in findings)) or "clean")

### The four

CA, FL, NY and TX declare terrorism territory against a state-specific code rather than the ZIP-derived one that eleven other jurisdictions use. It cannot be derived from a ZIP.

In [ ]:
print("PLACE_CODED:", V.PLACE_CODED)
print()
print("This constant is quoted in three places -- here, the E8 escalation and R22 --")
print("and OI-91 is open precisely because a second measurement disagrees with it.")

## What it refuses

Nothing. That is the design: it walks the whole submission and returns everything it found.

In [ ]:
print("validate() returns:", type(findings).__name__)
print("it raises on a bad submission?  No -- findings are data, not control flow.")
print()
print("The engine's own refusals are what stop a rating. See notebook 17.")

## Try it yourself

1. Add a field ISO doesn't declare to the payload. Which check fires, and at what severity?
2. Remove a required field. Does validation object, or does the engine?
3. Validate the same payload against `NY`'s schema. What changes, and why?

In [ ]:
# your turn